# 02 - Severity Analysis

ACNE04 grades each face into four ordered classes (**mild, moderate, severe, very_severe**), derived from Hayashi lesion-count criteria. Here we quantify the class distribution and imbalance - which drives class weighting / augmentation choices before training.

> Read-only.

In [1]:
# Make the src/ package importable from the notebooks/ folder (CPU-only, no install).
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
pd.set_option('display.max_columns', 30)

from acnevision.data import (
    eda_data, eda_images, eda_severity, eda_lesions, eda_boxes, eda_viz,
)

FIG = Path.cwd().parent / 'outputs' / 'figures'


In [2]:
# Load once: per-image table (+ real image sizes) and the per-box table.
df = eda_data.load_dataframe()
df = eda_data.attach_image_sizes(df)
boxes = eda_data.explode_boxes(df)
print('images:', len(df), '| lesion boxes:', len(boxes))
df.head(3)


images: 1457 | lesion boxes: 18983


,image_file,file_name,level,severity,lesion_count_label,num_lesions_annotated,lesion_boxes,split,fold_0_split,fold_1_split,fold_2_split,fold_3_split,fold_4_split,num_boxes,image_path,width,height,mode,img_format,missing,corrupted,aspect_ratio
0,acne_00000.jpg,levle0_0.jpg,0,mild,3,3,"{'xmin': [223, 217, 186], 'ymin': [290, 250, 5...",train,train,train,train,test,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,525,700,RGB,JPEG,False,False,0.750000
1,acne_00001.jpg,levle0_1.jpg,0,mild,3,3,"{'xmin': [394, 654, 930], 'ymin': [933, 2160, ...",train,train,train,test,train,train,3,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,3112,3456,RGB,JPEG,False,False,0.900463
2,acne_00002.jpg,levle0_10.jpg,1,moderate,20,20,"{'xmin': [357, 394, 314, 316, 265, 170, 203, 9...",train,train,train,train,test,train,20,/media/mahabur/af33425f-321f-4960-a4d5-f823d20...,478,460,RGB,JPEG,False,False,1.039130


## Class distribution table (count, %, imbalance ratio)
`imbalance_ratio` = majority-class count / this-class count.

In [3]:
table = eda_severity.severity_table(df)
table

,severity,count,percentage,imbalance_ratio
0,mild,513,35.21,1.23
1,moderate,633,43.45,1.00
2,severe,182,12.49,3.48
3,very_severe,129,8.85,4.91


## Imbalance summary

In [4]:
eda_severity.imbalance_summary(table)

{'num_classes': 4,
 'largest_class': 'moderate',
 'smallest_class': 'very_severe',
 'max_imbalance_ratio': 4.91,
 'minority_percentage': 8.85}

## Consistency check: does numeric `level` match text `severity`?

In [5]:
mismatch = eda_severity.level_severity_consistency(df)
print('level<->severity mismatches:', len(mismatch))
mismatch.head()

level<->severity mismatches: 0


,image_file,level,severity


## Bar chart (saved under outputs/figures/)

In [6]:
eda_viz.plot_severity_bar(table, FIG / 'severity_distribution_bar.png')
print('saved severity bar chart')

saved severity bar chart


**Takeaways:** moderate (43%) and mild (35%) dominate; severe (12%) and very_severe (9%) are the minority. Max imbalance ~4.9x. Use class weights or targeted augmentation for the two minority classes - never fabricate labels.